In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
from tqdm import tqdm
import numpy as np
import plotly.graph_objects as go

In [3]:
from autoeval_utils import fetch_answer, get_label_rouge

In [4]:
# Load the data for answers
eval_data_path = "../../data/HintGenDataset/valid.json"

eval_data = json.load(open(eval_data_path, "r"))
answers = [d["answer"] for d in eval_data]

In [5]:
output_dir = 'output/information_gain_study'
MODEL_NAMES = ['google/gemma-2-2b', 'google/gemma-2-9b', \
               'meta-llama/Llama-3.2-1B-Instruct', 'meta-llama/Llama-3.2-3B-Instruct', 'meta-llama/Llama-3.1-8B-Instruct', \
               'Qwen/Qwen3-0.6B', 'Qwen/Qwen3-1.7B', 'Qwen/Qwen3-4B', 'Qwen/Qwen3-8B' \
               'Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct', 'Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen2.5-7B-Instruct', \
               'Qwen/Qwen2.5-0.5B', 'Qwen/Qwen2.5-1.5B', 'Qwen/Qwen2.5-3B', 'Qwen/Qwen2.5-7B' \
               'mistralai/Mistral-7B-Instruct-v0.2'
            ]
MAX_CHAIN_SIZE = 30

predictions = {}
for model_name in MODEL_NAMES:
    predictions[model_name] = json.load(open(os.path.join(output_dir, f"{model_name.split('/')[-1]}_maxchain-{MAX_CHAIN_SIZE}.json")))

In [6]:
# evaluate the informativeness gain for the generated answers
gains = {}
for model_name in MODEL_NAMES:

    print(f"Evaluating for model {model_name}")

    gains[model_name] = {'r1_mean': [0] * MAX_CHAIN_SIZE, 'r1_max': [0] * MAX_CHAIN_SIZE, 
                         'r2_mean': [0] * MAX_CHAIN_SIZE, 'r2_max': [0] * MAX_CHAIN_SIZE, 
                         'rL_mean': [0] * MAX_CHAIN_SIZE, 'rL_max': [0] * MAX_CHAIN_SIZE, 
                        #  'bertscore_mean': [], 'bertscore_max': []
        }

    for inst_answer, inst_predictions in tqdm(zip(answers, predictions[model_name]), total=len(answers)):

        rouge_scores, nli_scores, bertscore_scores = {}, {}, {}

        # for the qa_response scores
        rouge_scores_qa = get_label_rouge(inst_answer, inst_predictions[0])

        # bertscores = get_label_bertscore([inst_answer] * len(inst_predictions), inst_predictions)
        # bertscore_scores_qa = bertscores[0]
        # bertscore_scores_chain = bertscores[1:]

        # for the chain scores
        rouge_scores_chain = []
        for chain in inst_predictions[1:]:
            rouge_scores_chain.append(get_label_rouge(inst_answer, chain))
        
        # find the avg and max gains for each metric
        r1_gains = [r1 - rouge_scores_qa[0] for r1, _, _ in rouge_scores_chain]
        r2_gains = [r2 - rouge_scores_qa[1] for _, r2, _ in rouge_scores_chain]
        rL_gains = [rL - rouge_scores_qa[2] for _, _, rL in rouge_scores_chain]
        # epsilon = 1e-5
        # r1_gains = [(r1 - rouge_scores_qa[0]) / max(rouge_scores_qa[0], epsilon) for r1, _, _ in rouge_scores_chain]
        # r2_gains = [(r2 - rouge_scores_qa[1]) / max(rouge_scores_qa[1], epsilon) for _, r2, _ in rouge_scores_chain]
        # rL_gains = [(rL - rouge_scores_qa[2]) / max(rouge_scores_qa[2], epsilon) for _, _, rL in rouge_scores_chain]

        # bertscore_gains = [bertscore - bertscore_scores_qa for bertscore in bertscore_scores_chain]

        gains[model_name]['r1_mean'] = [sum(x) for x in zip(gains[model_name]['r1_mean'], r1_gains)]
        gains[model_name]['r2_mean'] = [sum(x) for x in zip(gains[model_name]['r2_mean'], r2_gains)]
        gains[model_name]['rL_mean'] = [sum(x) for x in zip(gains[model_name]['rL_mean'], rL_gains)]
        # gains['nli_mean'].append(np.mean(nli_gains))
        # gains['nli_max'].append(np.max(nli_gains))
        # gains['bertscore_mean'].append(np.mean(bertscore_gains))
        # gains['bertscore_max'].append(np.max(bertscore_gains))

Evaluating for model google/gemma-2-2b


100%|██████████| 1000/1000 [00:06<00:00, 159.64it/s]


Evaluating for model google/gemma-2-9b


100%|██████████| 1000/1000 [00:11<00:00, 87.72it/s]


Evaluating for model meta-llama/Llama-3.2-1B-Instruct


...

100%|██████████| 1000/1000 [00:08<00:00, 117.95it/s]


Evaluating for model Qwen/Qwen2.5-3B


100%|██████████| 1000/1000 [00:08<00:00, 113.72it/s]


Evaluating for model Qwen/Qwen2.5-7Bmistralai/Mistral-7B-Instruct-v0.2


100%|██████████| 1000/1000 [00:11<00:00, 88.45it/s]


In [7]:
# normalize the gains by the number of instances
for model_name in MODEL_NAMES:
    for metric in ['r1_mean', 'r2_mean', 'rL_mean']:
        gains[model_name][metric] = [x / len(answers) for x in gains[model_name][metric]]

In [13]:
for metric in ['r1', 'r2', 'rL']:
    # plot the gains for each model
    fig = go.Figure()
    for model_name in MODEL_NAMES:
        fig.add_trace(go.Scatter
        (
            x = list(range(1, len(gains[model_name][f'{metric}_mean']) + 1)),
            y = gains[model_name][f'{metric}_mean'],
            mode = 'lines',
            name = f"{model_name.split('/')[-1]}"
        ))

    fig.update_layout(
        # title = f"Informativeness gain for different models ({metric})",
                        xaxis_title = "length of hint chain",
                        yaxis_title = "Gain",
                        legend_title = "Models",
                        width = 1000,
                        height = 400,
                        margin=dict(l=20, r=20, t=30, b=20),
                        showlegend = True,
                        plot_bgcolor='white',         # Set plot background to white
                        paper_bgcolor='white',        # Set paper background to white
    )
    fig.write_image(f"paper_figs/info_gain_model_selection.pdf", engine="kaleido")
    fig.show()

In [ ]:
len(gains[model_name]['r1_mean']), len(scores[model_name]['r1_mean'])

NameError: name 'scores' is not defined

In [ ]:
for metric in ['r1', 'r2', 'rL']:
    # plot the gains for each model
    fig = go.Figure()
    for model_name in MODEL_NAMES:
        fig.add_trace(go.Scatter
        (
            x = list(range(1, len(gains[model_name][f'{metric}_mean']) + 1)),
            y = gains[model_name][f'{metric}_mean'],
            mode = 'lines',
            name = f"{model_name.split('/')[-1]}"
        ))

    fig.update_layout(title = f"Informativeness gain for different models ({metric})",
                        xaxis_title = "Instance",
                        yaxis_title = "Gain",
                        legend_title = "Models",
                        width = 1000,
                        showlegend = True)
    fig.show()

In [ ]:
# evaluate the informativeness gain for the generated answers
scores = {}
for model_name in MODEL_NAMES:

    print(f"Evaluating for model {model_name}")

    scores[model_name] = {'r1_mean': [0] * (MAX_CHAIN_SIZE+1),
                         'r2_mean': [0] * (MAX_CHAIN_SIZE+1),
                         'rL_mean': [0] * (MAX_CHAIN_SIZE+1),
        }

    for inst_answer, inst_predictions in tqdm(zip(answers, predictions[model_name]), total=len(answers)):
        # for the chain scores
        rouge_scores = []
        for prediction in inst_predictions:
            rouge_scores.append(get_label_rouge(inst_answer, prediction))
        
        # find the scores for each metric
        r1_scores = [r1 for r1, _, _ in rouge_scores]
        r2_scores = [r2 for _, r2, _ in rouge_scores]
        rL_scores = [rL for _, _, rL in rouge_scores]

        scores[model_name]['r1_mean'] = [sum(x) for x in zip(scores[model_name]['r1_mean'], r1_scores)]
        scores[model_name]['r2_mean'] = [sum(x) for x in zip(scores[model_name]['r2_mean'], r2_scores)]
        scores[model_name]['rL_mean'] = [sum(x) for x in zip(scores[model_name]['rL_mean'], rL_scores)]

Evaluating for model google/gemma-2-2b


 74%|███████▍  | 740/1000 [00:04<00:01, 157.71it/s]

100%|██████████| 1000/1000 [00:06<00:00, 156.45it/s]


Evaluating for model google/gemma-2-9b


100%|██████████| 1000/1000 [00:11<00:00, 84.89it/s]


...

100%|██████████| 1000/1000 [00:08<00:00, 116.37it/s]


Evaluating for model Qwen/Qwen2.5-3B


100%|██████████| 1000/1000 [00:08<00:00, 112.72it/s]


Evaluating for model Qwen/Qwen2.5-7Bmistralai/Mistral-7B-Instruct-v0.2


100%|██████████| 1000/1000 [00:11<00:00, 87.41it/s]


In [ ]:
# normalize the scores by the number of instances
for model_name in MODEL_NAMES:
    for metric in ['r1_mean', 'r2_mean', 'rL_mean']:
        scores[model_name][metric] = [x / len(answers) for x in scores[model_name][metric]]

In [ ]:
for metric in ['r1', 'r2', 'rL']:
    # plot the scores for each model
    fig = go.Figure()
    for model_name in MODEL_NAMES:
        fig.add_trace(go.Scatter
        (
            x = list(range(1, len(scores[model_name][f'{metric}_mean']) + 1)),
            y = scores[model_name][f'{metric}_mean'],
            mode = 'lines',
            name = f"{model_name.split('/')[-1]}"
        ))

    fig.update_layout(title = f"ROUGE ({metric}-recall) for different models",
                        xaxis_title = "Instance",
                        yaxis_title = "ROUGE score",
                        legend_title = "Models",
                        width = 1000,
                        showlegend = True)
    # save the plot
    fig.show()


### Basic QA capabilities

In [ ]:
# evaluate the informativeness gain for the generated answers
qa_scores = {}
for model_name in MODEL_NAMES:

    print(f"Evaluating for model {model_name}")

    qa_scores[model_name] = {'r1': [], 'r2': [], 'rL': []}

    for inst_answer, inst_predictions in tqdm(zip(answers, predictions[model_name]), total=len(answers)):

        # for the qa_response scores
        rouge_scores_qa = get_label_rouge(inst_answer, inst_predictions[0])
        qa_scores[model_name]['r1'].append(rouge_scores_qa[0])
        qa_scores[model_name]['r2'].append(rouge_scores_qa[1])
        qa_scores[model_name]['rL'].append(rouge_scores_qa[2])
    
# normalize the scores by the number of instances
for model_name in MODEL_NAMES:
    for metric in ['r1', 'r2', 'rL']:
        qa_scores[model_name][metric] = np.mean(qa_scores[model_name][metric])

for metric in ['r1', 'r2', 'rL']:
    # plot the scores for each model
    fig = go.Figure()
    for model_name in MODEL_NAMES:
        fig.add_trace(go.Bar
        (
            x = [model_name.split('/')[-1]],
            y = [qa_scores[model_name][metric]],
            name = f"{model_name.split('/')[-1]}"
        ))

    fig.update_layout(title = f"ROUGE ({metric}-recall) for different models (QA response)",
                        xaxis_title = "Model",
                        yaxis_title = "ROUGE score",
                        legend_title = "Models",
                        width = 1000,
                        showlegend = True)
    # save the plot
    fig.show()

Evaluating for model google/gemma-2-2b


100%|██████████| 1000/1000 [00:00<00:00, 12153.53it/s]


Evaluating for model google/gemma-2-9b


 42%|████▏     | 417/1000 [00:00<00:00, 4167.32it/s]

100%|██████████| 1000/1000 [00:00<00:00, 3826.27it/s]


...

100%|██████████| 1000/1000 [00:00<00:00, 4899.78it/s]


Evaluating for model Qwen/Qwen2.5-3B


100%|██████████| 1000/1000 [00:00<00:00, 4039.54it/s]


Evaluating for model Qwen/Qwen2.5-7Bmistralai/Mistral-7B-Instruct-v0.2


100%|██████████| 1000/1000 [00:00<00:00, 2626.63it/s]


### Plot trade-off of memorization and reasoning for this task

In [40]:
# Taking qa_scores as x-axis, and average gains as y-axis
for metric in ['r1', 'r2', 'rL']:
    # plot the gains for each model
    fig = go.Figure()
    for model_name in MODEL_NAMES:
        fig.add_trace(go.Scatter
        (
            x = [qa_scores[model_name][metric]],
            y = [np.mean(gains[model_name][f'{metric}_mean'])],
            mode = 'markers',
            name = f"{model_name.split('/')[-1]}"
        ))

    fig.update_layout(title = f"Informativeness gain vs ROUGE ({metric}-recall) for different models",
                        xaxis_title = "ROUGE score (QA response)",
                        yaxis_title = "Informativeness gain",
                        legend_title = "Models",
                        width = 1000,
                        showlegend = True)
    fig.show()

In [67]:
for metric in ['r1', 'r2', 'rL']:
    # plot the metric scores for each model
    fig = go.Figure()
    for model_name in MODEL_NAMES:
        fig.add_trace(go.Scatter(
            x = [qa_scores[model_name][metric]],
            y = [np.mean(scores[model_name][f'{metric}_mean'])],
            mode = 'markers+text',
            name = f"{model_name.split('/')[-1]}",
            text = [model_name.split('/')[-1]],           # Add model name as label
            textposition = 'top center'                   # Position the label
        ))
    x_val = [qa_scores[model_name][metric] for model_name in MODEL_NAMES]
    y_val = [np.mean(scores[model_name][f'{metric}_mean']) for model_name in MODEL_NAMES]

    x_min = max(0, min(x_val) - 0.08)
    x_max = min(1, max(x_val) + 0.08)
    y_min = max(0, min(y_val) - 0.08)
    y_max = min(1, max(y_val) + 0.08)

    fig.update_layout(
        title = f"Informativeness score vs ROUGE ({metric}-recall) for different models",
        xaxis_title = "ROUGE score (QA response)",
        yaxis_title = "Avg. informativeness score",
        width = 800,
        height = 600,
        showlegend = False,
        plot_bgcolor='white',         # Set plot background to white
        paper_bgcolor='white',        # Set paper background to white
        xaxis=dict(
            showgrid=True,
            gridcolor='lightgray',
            range=[x_min, x_max]   # Add margin inside x-axis
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor='lightgray',
            range=[y_min, y_max]   # Add margin inside x-axis
        )
    )

    # Collect points
    points = []
    for model_name in MODEL_NAMES:
        x = qa_scores[model_name][metric]
        y = np.mean(scores[model_name][f'{metric}_mean'])
        points.append((x, y, model_name))

    # Find Pareto optimal points (lower x, higher y is better)
    pareto_points = []
    for i, (x0, y0, name0) in enumerate(points):
        dominated = False
        for j, (x1, y1, name1) in enumerate(points):
            if (x1 < x0 and y1 > y0):
                dominated = True
                break
        if not dominated:
            pareto_points.append((x0, y0, name0))

    # Sort Pareto points by x (ascending)
    pareto_points = sorted(pareto_points, key=lambda tup: tup[0])

    # Add Pareto front line
    fig.add_trace(go.Scatter(
        x=[pt[0] for pt in pareto_points],
        y=[pt[1] for pt in pareto_points],
        mode='lines+markers',
        line=dict(color='red', dash='dash'),
        marker=dict(symbol='circle', size=8, color='red'),
        name='Pareto Front'
    ))

    fig.show()

In [16]:
for metric in ['r1', 'r2', 'rL']:
    # plot the metric gain for each model
    fig = go.Figure()
    for model_name in MODEL_NAMES:
        fig.add_trace(go.Scatter(
            x = [qa_scores[model_name][metric]],
            y = [np.mean(gains[model_name][f'{metric}_mean'])],
            mode = 'markers+text',
            name = f"{model_name.split('/')[-1]}",
            text = [model_name.split('/')[-1]],           # Add model name as label
            textposition = 'top center'                   # Position the label
        ))
    x_val = [qa_scores[model_name][metric] for model_name in MODEL_NAMES]
    y_val = [np.mean(gains[model_name][f'{metric}_mean']) for model_name in MODEL_NAMES]

    x_min = max(-1, min(x_val) - 0.08)
    x_max = min(1, max(x_val) + 0.08)
    y_min = max(-1, min(y_val) - 0.08)
    y_max = min(1, max(y_val) + 0.08)

    fig.update_layout(
        title = f"Informativeness score vs ROUGE ({metric}-recall) for different models",
        xaxis_title = "ROUGE score (QA response)",
        yaxis_title = "Avg. informativeness gain",
        width = 800,
        height = 600,
        showlegend = False,
        plot_bgcolor='white',         # Set plot background to white
        paper_bgcolor='white',        # Set paper background to white
        xaxis=dict(
            showgrid=True,
            gridcolor='lightgray',
            range=[x_min, x_max]   # Add margin inside x-axis
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor='lightgray',
            range=[y_min, y_max]   # Add margin inside x-axis
        )
    )

    # Collect points
    points = []
    for model_name in MODEL_NAMES:
        x = qa_scores[model_name][metric]
        y = np.mean(gains[model_name][f'{metric}_mean'])
        points.append((x, y, model_name))

    # Find Pareto optimal points (lower x, higher y is better)
    pareto_points = []
    for i, (x0, y0, name0) in enumerate(points):
        dominated = False
        for j, (x1, y1, name1) in enumerate(points):
            if (x1 < x0 and y1 > y0):
                dominated = True
                break
        if not dominated:
            pareto_points.append((x0, y0, name0))

    # Sort Pareto points by x (ascending)
    pareto_points = sorted(pareto_points, key=lambda tup: tup[0])

    # Add Pareto front line
    fig.add_trace(go.Scatter(
        x=[pt[0] for pt in pareto_points],
        y=[pt[1] for pt in pareto_points],
        mode='lines+markers',
        line=dict(color='red', dash='dash'),
        marker=dict(symbol='circle', size=8, color='red'),
        name='Pareto Front'
    ))

    fig.show()

In [46]:
# Taking qa_scores as x-axis, and average scores as y-axis
for metric in ['r1', 'r2', 'rL']:
    # plot the metric scores for each model
    fig = go.Figure()
    for model_name in MODEL_NAMES:
        fig.add_trace(go.Scatter
        (
            x = [qa_scores[model_name][metric]],
            y = [np.mean(scores[model_name][f'{metric}_mean'])],
            mode = 'markers',
            name = f"{model_name.split('/')[-1]}"
        ))

    fig.update_layout(title = f"Informativeness score vs ROUGE ({metric}-recall) for different models",
                        xaxis_title = "ROUGE score (QA response)",
                        yaxis_title = "Avg. informativeness score",
                        legend_title = "Models",
                        width = 1000,
                        showlegend = True)
    fig.show()